<a href="https://colab.research.google.com/github/edwardsnj/glygen-colab-notebooks/blob/main/uniprotkb_nlinked_sites/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
#
# Import the modules from the github repository
#
import httpimport

with httpimport.github_repo("edwardsnj","glygen-colab-notebooks", ref="main"):
  from glygen import GlyGenDownloader


In [38]:
# Download UniProtKB protein site annotations for each species
ggdl = GlyGenDownloader()

template = "*_protein_site_annotation_uniprotkb.csv"
FILES = ggdl.filenames(template)

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","ref_aa","ann_type","annotation","xref_key","xref_id"],
  "notna": ["start_pos","xref_id"],
  "asint": ["start_pos"],
  "addfilename": True,
  "transform": {"species": lambda df: (df['filename'].str.split("_", expand=True))[0]},
  "dropcols": ["filename"],
  "dropdups": True,
}
uniprotkb_site_annotations = ggdl.dataframe(FILES,**params)


Using cached arabidopsis_protein_site_annotation_uniprotkb.csv (5.93 MB).
Using cached bovine_protein_site_annotation_uniprotkb.csv (3.91 MB).
Using cached chicken_protein_site_annotation_uniprotkb.csv (1.40 MB).
Using cached dicty_protein_site_annotation_uniprotkb.csv (672.98 KB).
Using cached fruitfly_protein_site_annotation_uniprotkb.csv (3.04 MB).
Using cached hamster_protein_site_annotation_uniprotkb.csv (752.29 KB).
Using cached hcv1a_protein_site_annotation_uniprotkb.csv (31.71 KB).
Using cached hcv1b_protein_site_annotation_uniprotkb.csv (31.34 KB).
Using cached human_protein_site_annotation_uniprotkb.csv (53.79 MB).
Using cached mouse_protein_site_annotation_uniprotkb.csv (15.80 MB).
Using cached pig_protein_site_annotation_uniprotkb.csv (1.70 MB).
Using cached rat_protein_site_annotation_uniprotkb.csv (7.69 MB).
Using cached sarscov1_protein_site_annotation_uniprotkb.csv (68.27 KB).
Using cached sarscov2_protein_site_annotation_uniprotkb.csv (139.28 KB).
Using cached yeast_pr

In [39]:
# Select N-linked GlcNAc Asn sites with PubMed ids
nlinked_rows = uniprotkb_site_annotations["annotation"].str.contains("N-linked")
pubmed_rows = uniprotkb_site_annotations["xref_key"]=="protein_xref_pubmed"
asn_rows = uniprotkb_site_annotations['ref_aa']=="N"
nlinked_sites_pubmed = uniprotkb_site_annotations[nlinked_rows&pubmed_rows&asn_rows]

nlinked_sites_pubmed

,uniprotkb_canonical_ac,ann_type,start_pos,annotation,ref_aa,xref_key,xref_id,species
1912,O64743-1,Glycosylation_Annotation,431,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,26037923,arabidopsis
1915,O64743-1,Glycosylation_Annotation,57,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,26037923,arabidopsis
2813,P24806-1,Glycosylation_Annotation,106,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,10406121,arabidopsis
4868,Q9FL28-1,Glycosylation_Annotation,94,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,24114786,arabidopsis
4870,Q9FL28-1,Glycosylation_Annotation,388,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,24114786,arabidopsis
...,...,...,...,...,...,...,...,...
626731,F8W463-1,Glycosylation_Annotation,113,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,22535247,zebrafish
626732,F8W463-1,Glycosylation_Annotation,113,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,19641588,zebrafish
626734,F8W463-1,Glycosylation_Annotation,187,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,19641588,zebrafish
626736,F8W463-1,Glycosylation_Annotation,213,N-linked (GlcNAc...) asparagine,N,protein_xref_pubmed,19641588,zebrafish


In [40]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Plot histogram of frequency of sites in a pubmed id (partition by species)
counts = pd.DataFrame(nlinked_sites_pubmed[["xref_id","species"]].value_counts())
counts.reset_index("species", inplace=True)
counts = counts[counts["count"] >= 10]

import plotly.express as px

fig = px.histogram(counts, x="count", color="species")
fig.layout.xaxis.title = "Number of sites ( >= 10 )"
fig.layout.yaxis.title = "Number of pubmed ids"
fig.update_layout(bargap=0.1)
fig.update_traces(xbins=dict(start=0, end=1000, size=50))
fig.show()

# sns.histplot(data=counts, x='count', hue="species", multiple="stack")
# plt.xlabel("Number of sites ( >= 10 )")
# plt.ylabel("Number of pubmed ids")
# plt.show()

In [41]:
counts[counts["count"]>=100]

,species,count
xref_id,,
19159218,human,876
16335952,human,537
19349973,mouse,334
19656770,mouse,180
19349973,human,171
24090084,rat,129
19139490,human,128
16944957,mouse,110
12754519,human,107
